# Task 6 — System Demonstration
## Multimodal Authentication & Product Recommendation Pipeline

**Team:** Data Preprocessing Group  
**Author:** Kelvin (Task 6 — Integration & Demonstration)

---

### What This Notebook Does
Connects all teammates' work into a single, running end-to-end demo:

| Stage | Model | Source |
|---|---|---|
| Face Recognition | `facial_recognition_model.pkl` | Task 4 |
| Product Recommendation | `product_recommendation_model.pkl` | Task 4 |
| Voice Verification | `voiceprint_model.pkl` | Task 4 |

### Pipeline (from team flowchart)
```
User Input
    ↓
[STEP 1] Face Recognition  →  FAIL → Access Denied
    ↓ PASS
[STEP 2] Product Recommendation  (computed, held until voice passes)
    ↓
[STEP 3] Voice Verification  →  FAIL → Access Denied
    ↓ PASS
Display: Welcome + Recommended Product
```


## Cell 1 — Imports

In [26]:
import warnings
import numpy as np
import pandas as pd
import joblib
import os

warnings.filterwarnings('ignore')

# ── Auto-detect project root ─────────────────────────────────────────────────
# This works whether you open the notebook from the root folder
# OR from inside the models/ subfolder
def find_dir(target_name, start='.', max_depth=4):
    """Walk upward and downward to find a directory by name."""
    start = os.path.abspath(start)
    # Search downward first
    for root, dirs, files in os.walk(start):
        depth = root.replace(start, '').count(os.sep)
        if depth > max_depth:
            break
        if target_name in dirs:
            return os.path.join(root, target_name)
    return None

# ── Find saved_models ─────────────────────────────────────────────────────────
MODELS_DIR = find_dir('saved_models')
if MODELS_DIR is None:
    # Fallback: common relative paths
    candidates = [
        'saved_models',
        os.path.join('models', 'saved_models'),
        os.path.join('..', 'saved_models'),
        os.path.join('..', 'models', 'saved_models'),
    ]
    for c in candidates:
        if os.path.isdir(c):
            MODELS_DIR = c
            break

if MODELS_DIR is None:
    print("❌ Could not find saved_models/ folder.")
    print("   Run task_4_model_creation.ipynb first to generate it.")
else:
    print(f"✅ Found saved_models at: {os.path.abspath(MODELS_DIR)}")

# ── Find data folders ─────────────────────────────────────────────────────────
def find_file(filename, start='.', max_depth=5):
    """Find a file by name anywhere in the project."""
    for root, dirs, files in os.walk(os.path.abspath(start)):
        depth = root.replace(os.path.abspath(start), '').count(os.sep)
        if depth > max_depth:
            break
        if filename in files:
            return os.path.join(root, filename)
    return None

IMAGE_CSV  = find_file('image_features.csv')
MERGED_CSV = find_file('merged_dataset.csv')
AUDIO_CSV  = find_file('audio_features.csv')

print(f"{'✅' if IMAGE_CSV  else '❌'} image_features.csv  : {IMAGE_CSV  or 'NOT FOUND'}")
print(f"{'✅' if MERGED_CSV else '❌'} merged_dataset.csv  : {MERGED_CSV or 'NOT FOUND'}")
print(f"{'⚠️' if not AUDIO_CSV else '✅'} audio_features.csv  : {AUDIO_CSV  or 'Not found (synthetic fallback will be used)'}")

print('\nLibraries loaded.')


✅ Found saved_models at: /home/school/ML_ALU/alu-machine_learning/formatives/data_preprocessing/models/saved_models
❌ image_features.csv  : NOT FOUND
❌ merged_dataset.csv  : NOT FOUND
⚠️ audio_features.csv  : Not found (synthetic fallback will be used)

Libraries loaded.


## Cell 2 — Paths & Member→Customer ID Mapping

In [41]:
# ── Member → Customer ID mapping ─────────────────────────────────────────────
MEMBER_TO_CUSTOMER_ID = {
    'Member_1': 'A192',
    'Member_2': 'A190',
    'Member_3': 'A150',
    'Member_4': 'A103',
}

ALL_MEMBERS = list(MEMBER_TO_CUSTOMER_ID.keys())

print('Member → Customer ID mapping:')
for member, cid in MEMBER_TO_CUSTOMER_ID.items():
    print(f'  {member:20s} → {cid}')

Member → Customer ID mapping:
  Member_1             → A192
  Member_2             → A190
  Member_3             → A150
  Member_4             → A103


## Cell 3 — Load All Models

In [42]:
# ── Verify all model files exist before loading ──────────────────────────────
REQUIRED_FILES = [
    'facial_recognition_model.pkl',
    'face_label_encoder.pkl',
    'voiceprint_model.pkl',
    'voice_label_encoder.pkl',
    'audio_scaler.pkl',
    'product_recommendation_model.pkl',
    'product_label_encoder.pkl',
    'product_feature_columns.pkl',
]

if MODELS_DIR is None:
    raise FileNotFoundError("saved_models/ folder not found. Run task_4_model_creation.ipynb first.")

missing = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(MODELS_DIR, f))]
if missing:
    print("❌ Missing model files:")
    for m in missing: print(f"   • {m}")
    print("\n➡  Run task_4_model_creation.ipynb first to generate these files.")
    raise FileNotFoundError(f"Missing {len(missing)} model file(s) in {MODELS_DIR}")

# ── Load all models ───────────────────────────────────────────────────────────
face_model   = joblib.load(os.path.join(MODELS_DIR, 'facial_recognition_model.pkl'))
face_enc     = joblib.load(os.path.join(MODELS_DIR, 'face_label_encoder.pkl'))
voice_model  = joblib.load(os.path.join(MODELS_DIR, 'voiceprint_model.pkl'))
voice_enc    = joblib.load(os.path.join(MODELS_DIR, 'voice_label_encoder.pkl'))
audio_scaler = joblib.load(os.path.join(MODELS_DIR, 'audio_scaler.pkl'))
prod_model   = joblib.load(os.path.join(MODELS_DIR, 'product_recommendation_model.pkl'))
prod_enc     = joblib.load(os.path.join(MODELS_DIR, 'product_label_encoder.pkl'))
prod_cols    = joblib.load(os.path.join(MODELS_DIR, 'product_feature_columns.pkl'))

print('✅ All models loaded successfully.')
print(f'   Models directory     : {os.path.abspath(MODELS_DIR)}')
print(f'   Face classes         : {list(face_enc.classes_)}')
print(f'   Voice classes        : {list(voice_enc.classes_)}')
print(f'   Product categories   : {list(prod_enc.classes_)}')


✅ All models loaded successfully.
   Models directory     : /home/school/ML_ALU/alu-machine_learning/formatives/data_preprocessing/models/saved_models
   Face classes         : ['David', 'Kelvin', 'Michael Kimani', 'Samuel']
   Voice classes        : ['David', 'Kelvin', 'Michael Kimani', 'Samuel']
   Product categories   : ['Books', 'Clothing', 'Electronics', 'Groceries', 'Sports']


## Cell 4 — Load Feature Data

In [ ]:
# ── Image features ────────────────────────────────────────────────────────────
IMAGE_CSV  = 'models/data/processed/image_features.csv'
MERGED_CSV = 'models/data/processed/merged_dataset.csv'
AUDIO_CSV  = 'models/features/audio_features.csv'

if not os.path.exists(IMAGE_CSV):
    raise FileNotFoundError(f"Not found: {IMAGE_CSV}")
image_df = pd.read_csv(IMAGE_CSV)
print(f'✅ Image features  : {image_df.shape}')
print(f'   Members found   : {list(image_df["member"].unique())}')

# ── Merged dataset ─────────────────────────────────────────────────────────────
if not os.path.exists(MERGED_CSV):
    raise FileNotFoundError(f"Not found: {MERGED_CSV}")
merged_df = pd.read_csv(MERGED_CSV)
bool_cols = merged_df.select_dtypes(include='bool').columns
merged_df[bool_cols] = merged_df[bool_cols].astype(int)
print(f'✅ Merged dataset  : {merged_df.shape}')

# ── Audio features ─────────────────────────────────────────────────────────────
if os.path.exists(AUDIO_CSV):
    audio_df = pd.read_csv(AUDIO_CSV)
    print(f'✅ Audio features  : {audio_df.shape}  [REAL DATA]')
else:
    print('⚠️  audio_features.csv not found — using synthetic fallback')

    # Member names MUST match image_df exactly
    AUDIO_COLS = [f'mfcc_{i}' for i in range(1, 14)] + \
                 ['spectral_rolloff', 'rms_energy', 'zcr']

    # Use ALL_MEMBERS which now contains Member_1 ... Member_4
    np.random.seed(42)
    member_centers = {m: np.random.randn(len(AUDIO_COLS)) * 10 for m in ALL_MEMBERS}

    rows = []
    for member in ALL_MEMBERS:
        center = member_centers[member]
        for phrase in ['yes_approve', 'confirm_transaction']:
            for stype in ['original', 'augmented']:
                noise = np.random.randn(len(AUDIO_COLS)) * 0.5
                row = {
                    'member'      : member,
                    'phrase_label': phrase,
                    'sample_type' : stype,
                    'file_name'   : f'{member}_{phrase}_{stype}.wav'
                }
                for j, col in enumerate(AUDIO_COLS):
                    row[col] = center[j] + noise[j]
                rows.append(row)
            for k in range(4):
                noise = np.random.randn(len(AUDIO_COLS)) * 0.5
                row = {
                    'member'      : member,
                    'phrase_label': phrase,
                    'sample_type' : 'augmented',
                    'file_name'   : f'{member}_{phrase}_aug{k}.wav'
                }
                for j, col in enumerate(AUDIO_COLS):
                    row[col] = center[j] + noise[j]
                rows.append(row)
    audio_df = pd.DataFrame(rows)
    print(f'✅ Synthetic audio : {audio_df.shape}')
    print(f'   Members in audio: {list(audio_df["member"].unique())}')

# ── Feature column lists ───────────────────────────────────────────────────────
FACE_FEATURES = [c for c in image_df.columns
                 if c not in ['member', 'expression', 'augmentation']]
AUDIO_FEATURES = [c for c in audio_df.columns
                  if c not in ['member', 'phrase_label', 'sample_type', 'file_name']]

print(f'\n   Face features  : {len(FACE_FEATURES)} columns')
print(f'   Audio features : {len(AUDIO_FEATURES)} columns')
print('✅ All data loaded.')

✅ Image features  : (108, 15) | Members: ['Member_1', 'Member_2', 'Member_3', 'Member_4']
✅ Merged dataset  : (213, 21)
⚠️  audio_features.csv not found — using synthetic fallback
   Synthetic audio shape: (48, 20)

   Face features  : 12 columns
   Audio features : 16 columns
✅ Data loaded successfully.


## Cell 5 — Customer Profile Helper

In [44]:
def get_customer_profile(customer_id: str):
    """
    Fetches the most recent transaction row for a given customer ID.
    Returns the row as a dict, or None if the customer ID is not found.
    """
    subset = merged_df[merged_df['customer_id_new'] == customer_id]
    if subset.empty:
        return None
    return subset.sort_values('purchase_month', ascending=False).iloc[0].to_dict()


# ── Verify all 4 members have valid profiles ────────────────────────────
print('Customer profile verification:')
print(f'{"Member":<20} {"Customer ID":<12} {"Last Category":<15} {"Amount":>8}  Status')
print('-' * 65)
for member, cid in MEMBER_TO_CUSTOMER_ID.items():
    profile = get_customer_profile(cid)
    if profile:
        print(f'{member:<20} {cid:<12} {profile["product_category"]:<15} ${profile["purchase_amount"]:>7.2f}  ✅')
    else:
        print(f'{member:<20} {cid:<12} {"N/A":<15} {"N/A":>9}  ❌ NOT FOUND')


Customer profile verification:
Member               Customer ID  Last Category     Amount  Status
-----------------------------------------------------------------
Member_1             A192         Books           $ 491.00  ✅
Member_2             A190         Sports          $ 401.00  ✅
Member_3             A150         Sports          $ 389.00  ✅
Member_4             A103         Sports          $ 209.00  ✅


## Cell 6 — Core Pipeline Function

In [45]:
def run_pipeline(face_member: str, audio_member: str, label: str = 'Simulation'):
    """
    Runs the full 3-stage authentication + recommendation pipeline.
    
    Parameters
    ----------
    face_member  : str  Name of the member whose face features to use
    audio_member : str  Name of the member whose voice features to use
    label        : str  Description label for this simulation run
    
    Stage order (matches team flowchart):
      1. Face Recognition
      2. Product Recommendation  (runs after face, displayed after voice)
      3. Voice Verification
    """
    print(f'\n{"=" * 60}')
    print(f'  {label}')
    print(f'{"=" * 60}')

    # ── STAGE 1: Face Recognition ─────────────────────────────────────────
    print('\n[STAGE 1]  FACE RECOGNITION')
    face_row   = image_df[
        (image_df['member'] == face_member) &
        (image_df['augmentation'] == 'original')
    ].iloc[0]
    face_input = face_row[FACE_FEATURES].values.reshape(1, -1)
    face_pred  = face_model.predict(face_input)[0]
    face_proba = face_model.predict_proba(face_input).max()
    face_name  = face_enc.inverse_transform([face_pred])[0]

    print(f'  Identified as : {face_name}')
    print(f'  Confidence    : {face_proba:.1%}')

    if face_proba < 0.50:
        print('  ❌  ACCESS DENIED — face not recognized.')
        print(f'{"=" * 60}\n')
        return

    print('  ✅  Face recognized. Moving to product recommendation...')

    # ── STAGE 2: Product Recommendation ──────────────────────────────────
    print('\n[STAGE 2]  PRODUCT RECOMMENDATION')
    customer_id = MEMBER_TO_CUSTOMER_ID.get(face_name)
    profile     = get_customer_profile(customer_id)

    if profile is None:
        print(f'  ❌  No profile found for customer {customer_id}.')
        print(f'{"=" * 60}\n')
        return

    prod_input  = pd.DataFrame([profile])[prod_cols].values.astype(float)
    prod_pred   = prod_model.predict(prod_input)[0]
    prod_proba  = prod_model.predict_proba(prod_input).max()
    recommended = prod_enc.inverse_transform([prod_pred])[0]

    print(f'  Customer ID        : {customer_id}')
    print(f'  Predicted category : {recommended}')
    print(f'  Confidence         : {prod_proba:.1%}')
    print('  ✅  Recommendation ready. Moving to voice verification...')

    # ── STAGE 3: Voice Verification ───────────────────────────────────────
    print('\n[STAGE 3]  VOICE VERIFICATION')
    audio_row   = audio_df[
        (audio_df['member'] == audio_member) &
        (audio_df['sample_type'] == 'original')
    ].iloc[0]
    audio_input = audio_scaler.transform(
        audio_row[AUDIO_FEATURES].values.reshape(1, -1)
    )
    voice_pred  = voice_model.predict(audio_input)[0]
    voice_proba = voice_model.predict_proba(audio_input).max()
    voice_name  = voice_enc.inverse_transform([voice_pred])[0]

    print(f'  Voice identified  : {voice_name}')
    print(f'  Confidence        : {voice_proba:.1%}')

    if voice_name != face_name:
        print(f'  ❌  ACCESS DENIED — voice ({voice_name}) does not match face ({face_name}).')
        print(f'{"=" * 60}\n')
        return

    print('  ✅  Voice verified.')

    # ── FINAL OUTPUT ──────────────────────────────────────────────────────
    print(f'\n  🎉  AUTHENTICATION SUCCESSFUL')
    print(f'  Welcome, {face_name}!  (Customer ID: {customer_id})')
    print(f'  Recommended product category: {recommended}')
    print(f'{"=" * 60}\n')


print('Pipeline function defined and ready.')


Pipeline function defined and ready.


## Cell 7 — Simulation 1: Authorized Users (All 4 Members)

In [47]:
print('\n' + '█' * 60)
print('  SIMULATION 1 — AUTHORIZED USERS')
print('  Each member uses their OWN face and their OWN voice')
print('█' * 60)

for member in ['Member_1','Member_2','Member_3','Member_4']:
    run_pipeline(
        face_member  = member,
        audio_member = member,
        label        = f'Authorized User — {member}'
    )



████████████████████████████████████████████████████████████
  SIMULATION 1 — AUTHORIZED USERS
  Each member uses their OWN face and their OWN voice
████████████████████████████████████████████████████████████

  Authorized User — Member_1

[STAGE 1]  FACE RECOGNITION
  Identified as : Michael Kimani
  Confidence    : 100.0%
  ✅  Face recognized. Moving to product recommendation...

[STAGE 2]  PRODUCT RECOMMENDATION
  ❌  No profile found for customer None.


  Authorized User — Member_2

[STAGE 1]  FACE RECOGNITION
  Identified as : David
  Confidence    : 99.9%
  ✅  Face recognized. Moving to product recommendation...

[STAGE 2]  PRODUCT RECOMMENDATION
  ❌  No profile found for customer None.


  Authorized User — Member_3

[STAGE 1]  FACE RECOGNITION
  Identified as : Kelvin
  Confidence    : 100.0%
  ✅  Face recognized. Moving to product recommendation...

[STAGE 2]  PRODUCT RECOMMENDATION
  ❌  No profile found for customer None.


  Authorized User — Member_4

[STAGE 1]  FACE RECO

## Cell 8 — Simulation 2: Unauthorized — Face/Voice Mismatch

In [48]:
print('\n' + '█' * 60)
print('  SIMULATION 2 — UNAUTHORIZED ATTEMPTS')
print('  Attacker submits one person\'s face but a different voice')
print('█' * 60)

# Scenario A: Kelvin's face, Samuel's voice
run_pipeline(
    face_member  = 'Kelvin',
    audio_member = 'Samuel',
    label        = 'UNAUTHORIZED — Kelvin face + Samuel voice'
)

# Scenario B: David's face, Michael's voice
run_pipeline(
    face_member  = 'David',
    audio_member = 'Michael Kimani',
    label        = 'UNAUTHORIZED — David face + Michael Kimani voice'
)

# Scenario C: Michael's face, David's voice
run_pipeline(
    face_member  = 'Michael Kimani',
    audio_member = 'David',
    label        = 'UNAUTHORIZED — Michael Kimani face + David voice'
)



████████████████████████████████████████████████████████████
  SIMULATION 2 — UNAUTHORIZED ATTEMPTS
  Attacker submits one person's face but a different voice
████████████████████████████████████████████████████████████

  UNAUTHORIZED — Kelvin face + Samuel voice

[STAGE 1]  FACE RECOGNITION


IndexError: single positional indexer is out-of-bounds

## Cell 9 — Simulation 3: Unknown Face (Low Confidence)

In [ ]:
print('\n' + '█' * 60)
print('  SIMULATION 3 — UNKNOWN USER (STRANGER)')
print('  Random features that do not belong to any registered member')
print('█' * 60)

np.random.seed(99)
stranger_features = np.random.uniform(
    low  = image_df[FACE_FEATURES].min().values,
    high = image_df[FACE_FEATURES].max().values
)

face_input = stranger_features.reshape(1, -1)
face_pred  = face_model.predict(face_input)[0]
face_proba = face_model.predict_proba(face_input).max()
face_name  = face_enc.inverse_transform([face_pred])[0]

print(f'\n{"=" * 60}')
print(f'  UNAUTHORIZED — Unknown Stranger')
print(f'{"=" * 60}')
print(f'\n[STAGE 1]  FACE RECOGNITION')
print(f'  Best guess : {face_name}')
print(f'  Confidence : {face_proba:.1%}')

THRESHOLD = 0.60
if face_proba < THRESHOLD:
    print(f'  ❌  ACCESS DENIED — confidence {face_proba:.1%} is below threshold ({THRESHOLD:.0%}).')
    print('      Face not recognized as any registered member.')
else:
    print(f'  ✅  Recognized as {face_name} (this means stranger happened to match a member)')
print(f'{"=" * 60}\n')


## Cell 10 — Summary Report

In [ ]:
# ── Member → Customer ID mapping ────────────────────────────────────────────
# Each team member maps to a real customer ID in the merged dataset
MEMBER_TO_CUSTOMER_ID = {
    'David'         : 'A192',
    'Kelvin'        : 'A190',
    'Michael Kimani': 'A150',
    'Samuel'        : 'A103',
}

ALL_MEMBERS = list(MEMBER_TO_CUSTOMER_ID.keys())

print('Member → Customer ID mapping:')
for member, cid in MEMBER_TO_CUSTOMER_ID.items():
    print(f'  {member:20s} → {cid}')
